## Action Genome / STTran — Dataset & Prediction Statistics

This notebook reproduces the **metrics and plots** we built in the repo:

- **Frame-normalized frequencies** (presence per frame)
- **Assignment-normalized frequencies** (edge- and node-normalized)
- **Per-group predicate frequencies** (`@` attention, `^` spatial, `+` contact)
- **Per-predicate confidence stats** + **boxplots** per group
- **Threshold suggestions** derived from empirical mean/std
- **Sanity checks on GT person annotations** (`person_bbox.pkl` deep scan)

All computations are based on the logs under `STTran/output/logs/first5_videos/` (no model re-run).


In [13]:
import os
from pathlib import Path

# Jupyter rendering helpers
%matplotlib inline

# Paths (edit if your repo layout differs)
REPO_ROOT = Path.cwd().parent if (Path.cwd().name == "STTran") else Path.cwd()
STTRAN_DIR = REPO_ROOT / "STTran"
LOGS_DIR = STTRAN_DIR / "output" / "logs" / "first5_videos"
OUT_ROOT = STTRAN_DIR / "output" / "first5_videos"
SUMMARY_DIR = OUT_ROOT / "summary_plots"

print("REPO_ROOT:", REPO_ROOT)
print("LOGS_DIR:", LOGS_DIR)
print("OUT_ROOT:", OUT_ROOT)
print("SUMMARY_DIR:", SUMMARY_DIR)

assert LOGS_DIR.exists(), f"Missing logs dir: {LOGS_DIR}"

REPO_ROOT: /Users/tommasoaiello/Desktop/Magistrale/Secondo Semestre/Computer Vision/CV_Project
LOGS_DIR: /Users/tommasoaiello/Desktop/Magistrale/Secondo Semestre/Computer Vision/CV_Project/STTran/output/logs/first5_videos
OUT_ROOT: /Users/tommasoaiello/Desktop/Magistrale/Secondo Semestre/Computer Vision/CV_Project/STTran/output/first5_videos
SUMMARY_DIR: /Users/tommasoaiello/Desktop/Magistrale/Secondo Semestre/Computer Vision/CV_Project/STTran/output/first5_videos/summary_plots


In [14]:
# Import the same utilities used by our scripts
import sys
sys.path.insert(0, str(STTRAN_DIR))

from viz_terminal_scene_graphs import parse_terminal_log  # noqa

log_paths = sorted(LOGS_DIR.glob("*.log"))
print("#logs:", len(log_paths))
print("first 5:", [p.name for p in log_paths[:5]])

#logs: 29
first 5: ['00607.mp4.log', '00T1E.mp4.log', '015XE.mp4.log', '01THT.mp4.log', '02DPI.mp4.log']


## 1) Frequency plots (frame-normalized vs assignment-normalized)

We reuse the exact functions from `STTran/plot_log_frequencies.py`.

In [15]:
from plot_log_frequencies import (
    generate_frequency_plots,
    generate_predicate_frequency_plots_by_group,
    generate_assignment_normalized_plots,
)

SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

# Generate plots into a temp directory then move to SUMMARY_DIR (so notebook is idempotent)
_tmp = SUMMARY_DIR

pred_png, obj_png = generate_frequency_plots(logs_dir=str(LOGS_DIR), out_dir=str(_tmp))
att_png, spa_png, con_png = generate_predicate_frequency_plots_by_group(logs_dir=str(LOGS_DIR), out_dir=str(_tmp))
edge_all, edge_att, edge_spa, edge_con, node_obj = generate_assignment_normalized_plots(
    logs_dir=str(LOGS_DIR), out_dir=str(_tmp)
)

(pred_png, obj_png, att_png, spa_png, con_png, edge_all, edge_att, edge_spa, edge_con, node_obj)

('/Users/tommasoaiello/Desktop/Magistrale/Secondo Semestre/Computer Vision/CV_Project/STTran/output/first5_videos/summary_plots/predicate_frequency.png',
 '/Users/tommasoaiello/Desktop/Magistrale/Secondo Semestre/Computer Vision/CV_Project/STTran/output/first5_videos/summary_plots/object_frequency.png',
 '/Users/tommasoaiello/Desktop/Magistrale/Secondo Semestre/Computer Vision/CV_Project/STTran/output/first5_videos/summary_plots/predicate_frequency_attention.png',
 '/Users/tommasoaiello/Desktop/Magistrale/Secondo Semestre/Computer Vision/CV_Project/STTran/output/first5_videos/summary_plots/predicate_frequency_spatial.png',
 '/Users/tommasoaiello/Desktop/Magistrale/Secondo Semestre/Computer Vision/CV_Project/STTran/output/first5_videos/summary_plots/predicate_frequency_contact.png',
 '/Users/tommasoaiello/Desktop/Magistrale/Secondo Semestre/Computer Vision/CV_Project/STTran/output/first5_videos/summary_plots/predicate_frequency_edge_norm.png',
 '/Users/tommasoaiello/Desktop/Magistrale/S

In [16]:
# Display ALL generated plots inline (summary_plots/*.png)
# (Use IPython display to guarantee rendering in notebooks.)
from IPython.display import Image as IPyImage, display

pngs = sorted(SUMMARY_DIR.glob("*.png"))
print("#summary plots:", len(pngs))

# Put boxplots last so frequency plots come first
pngs = sorted(pngs, key=lambda p: ("boxplot" in p.name, p.name))

if not pngs:
    raise SystemExit(f"No PNGs found in {SUMMARY_DIR}")

for p in pngs:
    display(IPyImage(filename=str(p)))
    print(p.name)

#summary plots: 13


## 2) Predicate confidence statistics + boxplots

We generate:
- 3 boxplots (attention/spatial/contact)
- a per-predicate stats export (`CSV` + `JSON`) with empirical `mean/std` for confidence scores.

In [17]:
import subprocess

# Run the same script used previously (writes boxplots + predicate_confidence_stats.*)
# This keeps the notebook simple and guarantees parity with our CLI outputs.
subprocess.check_call([sys.executable, str(STTRAN_DIR / "plot_predicate_score_stats.py")])

# Move stats files into SUMMARY_DIR if they ended up in OUT_ROOT
for fname in ["predicate_confidence_stats.csv", "predicate_confidence_stats.json"]:
    p = OUT_ROOT / fname
    if p.exists():
        p.replace(SUMMARY_DIR / fname)

print("boxplots:",
      SUMMARY_DIR / "predicate_scores_attention_boxplot.png",
      SUMMARY_DIR / "predicate_scores_spatial_boxplot.png",
      SUMMARY_DIR / "predicate_scores_contact_boxplot.png")
print("stats:", SUMMARY_DIR / "predicate_confidence_stats.csv")

overall attention mean/std: 0.334019 0.377574
overall spatial   mean/std: 0.191521 0.091450
overall contact   mean/std: 0.089093 0.038450
wrote: /Users/tommasoaiello/Desktop/Magistrale/Secondo Semestre/Computer Vision/CV_Project/STTran/output/first5_videos/predicate_scores_attention_boxplot.png
wrote: /Users/tommasoaiello/Desktop/Magistrale/Secondo Semestre/Computer Vision/CV_Project/STTran/output/first5_videos/predicate_scores_spatial_boxplot.png
wrote: /Users/tommasoaiello/Desktop/Magistrale/Secondo Semestre/Computer Vision/CV_Project/STTran/output/first5_videos/predicate_scores_contact_boxplot.png
wrote: /Users/tommasoaiello/Desktop/Magistrale/Secondo Semestre/Computer Vision/CV_Project/STTran/output/first5_videos/predicate_confidence_stats.csv
wrote: /Users/tommasoaiello/Desktop/Magistrale/Secondo Semestre/Computer Vision/CV_Project/STTran/output/first5_videos/predicate_confidence_stats.json
boxplots: /Users/tommasoaiello/Desktop/Magistrale/Secondo Semestre/Computer Vision/CV_Proje

In [18]:
# Print per-category mean/std (as used for thresholds)
import json

stats = json.loads((SUMMARY_DIR / "predicate_confidence_stats.json").read_text())

def cat_stats(group: str):
    # Overall group stats across all edges in that group are printed by the script,
    # but we recompute here from the per-predicate rows for transparency.
    # We'll instead parse directly from the boxplot script outputs by re-reading logs.
    pass

# Recompute overall group mean/std across ALL edges in that group
import numpy as np
from collections import defaultdict

scores = {"att": [], "spatial": [], "contact": []}
for lp in log_paths:
    frames = parse_terminal_log(str(lp), topk_spatial=4, topk_contact=4)
    for fr in frames.values():
        for e in fr.edges:
            if e.group in scores:
                scores[e.group].append(float(e.score))

for g in ["att", "spatial", "contact"]:
    arr = np.asarray(scores[g], dtype=np.float32)
    print(g, "n_edges=", arr.size, "mean=", float(arr.mean()), "std=", float(arr.std()))

att n_edges= 4581 mean= 0.3340187668800354 std= 0.3775739073753357
spatial n_edges= 6106 mean= 0.19152143597602844 std= 0.09145041555166245
contact n_edges= 6106 mean= 0.0890926942229271 std= 0.03844967484474182


## 3) Thresholds

We often pick thresholds per group using a rule like:

- **tolerant**: \(\mu + 0.5\sigma\)
- **stricter**: \(\mu + 1.0\sigma\)

where \(\mu,\sigma\) are empirical mean/std of **confidence scores** for that group.

In [19]:
import numpy as np

mu = {g: float(np.mean(scores[g])) for g in scores}
sd = {g: float(np.std(scores[g])) for g in scores}

th_05 = {g: mu[g] + 0.5 * sd[g] for g in scores}
th_10 = {g: mu[g] + 1.0 * sd[g] for g in scores}

print("mu:", mu)
print("std:", sd)
print("threshold mu+0.5std:", th_05)
print("threshold mu+1.0std:", th_10)

mu: {'att': 0.33401877319362583, 'spatial': 0.1915214543072388, 'contact': 0.08909269570913855}
std: {'att': 0.3775738894526465, 'spatial': 0.0914504135285208, 'contact': 0.03844967600690291}
threshold mu+0.5std: {'att': 0.5228057179199491, 'spatial': 0.2372466610714992, 'contact': 0.10831753371259001}
threshold mu+1.0std: {'att': 0.7115926626462723, 'spatial': 0.2829718678357596, 'contact': 0.12754237171604146}


## 4) GT person annotation sanity check (deep scan)

Action Genome provides GT person boxes in `dataset/ag/annotations/person_bbox.pkl`.

This scan checks whether **any frame** contains more than one GT person box (i.e. `bbox` with shape `(2,4)` or larger).

In [20]:
import pickle
from collections import Counter
import numpy as np

AG_ROOT = REPO_ROOT / "dataset" / "ag"
PERSON_PKL = AG_ROOT / "annotations" / "person_bbox.pkl"

with open(PERSON_PKL, "rb") as f:
    person = pickle.load(f)

count_dist = Counter()
for k, v in person.items():
    b = v.get("bbox") if isinstance(v, dict) else v
    if b is None:
        n = 0
    else:
        arr = np.asarray(b)
        if arr.ndim == 2 and arr.shape[1] == 4:
            n = int(arr.shape[0])
        elif arr.ndim == 1 and arr.size >= 4:
            n = 1
        else:
            n = 0
    count_dist[n] += 1

print("Total frames in person_bbox.pkl:", sum(count_dist.values()))
print("Distribution (#persons -> #frames):")
for n in sorted(count_dist):
    print(f"  {n}: {count_dist[n]}")
print("Frames with >=2 persons:", sum(c for n,c in count_dist.items() if n>=2))

/var/folders/np/yvh5ndg15b56jryxv8dqc_x00000gn/T/ipykernel_55147/585671641.py:9: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  person = pickle.load(f)


Total frames in person_bbox.pkl: 288782
Distribution (#persons -> #frames):
  0: 40482
  1: 248300
Frames with >=2 persons: 0


## 5) (Optional) Show per-video plots

If you want, we can also render a grid of `edge_evolution.png` for each processed video (can be many images).

In [21]:
# Show per-video edge evolution plots (first N videos)
import matplotlib.pyplot as plt
from PIL import Image

N = 12  # change as needed
vid_dirs = sorted([p for p in OUT_ROOT.iterdir() if p.is_dir() and p.name.endswith('.mp4')])
edge_pngs = [p / 'edge_evolution.png' for p in vid_dirs if (p / 'edge_evolution.png').exists()]
edge_pngs = edge_pngs[:N]

print('edge evolution plots:', len(edge_pngs), 'of', len(vid_dirs), 'videos')

if edge_pngs:
    ncols = 2
    nrows = (len(edge_pngs) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, max(6, nrows * 5.5)))
    if nrows == 1:
        axes = [axes]  # type: ignore
    axes_flat = [ax for row in axes for ax in (row if isinstance(row, (list, tuple)) else [row])]

    for ax, p in zip(axes_flat, edge_pngs):
        ax.imshow(Image.open(p))
        ax.set_title(p.parent.name)
        ax.axis('off')

    for ax in axes_flat[len(edge_pngs):]:
        ax.axis('off')

    plt.tight_layout()

edge evolution plots: 12 of 28 videos


AttributeError: 'numpy.ndarray' object has no attribute 'imshow'